# 12.05 · RL with verifiable rewards (GRPO)

**Chapter:** Fine-tuning & Post-training · **Milestone:** M6

## Goals
- [ ] Why RL with verifiable rewards (RLVR) produced reasoning models: no labels needed, only a checker
- [ ] Write reward functions (correctness + format), including partial credit
- [ ] Run GRPO with TRL on a small model and read the reward curves
- [ ] Know the failure modes: reward hacking, zero-variance groups, slow generation

## Resources
- [DeepSeekMath / GRPO paper](https://arxiv.org/abs/2402.03300) · [DeepSeek-R1 paper](https://arxiv.org/abs/2501.12948) — free
- [TRL GRPOTrainer](https://huggingface.co/docs/trl/grpo_trainer) · [Unsloth – RL guide (free Colab notebooks)](https://docs.unsloth.ai/basics/reinforcement-learning-rl-guide) — free
- [Nathan Lambert – RLHF book, RLVR chapter](https://rlhfbook.com/) — free

**SFT** needs the right answer for each prompt. **RLVR** only needs a way to *check* an answer. The model samples several attempts (a group),
a program scores them, and GRPO pushes up the probability of better-than-group-average attempts (04.04's toy, now on a real model).
Here the checker is `task.score` — our extraction task is verifiable.

In [ ]:
import json
import time

import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import GRPOConfig, GRPOTrainer

from aieng import DATA
from task import dataset, messages, parse, score

FT = DATA / "ft"; FT.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
BASE = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(BASE)
test = dataset(60, seed=2)

## 1. Reward functions
TRL calls each reward function with the batch of completions (+ dataset columns as kwargs) and expects one float per completion.
Give **partial credit** so a mostly-right answer beats a useless one — otherwise early on every sample scores 0, the group has zero variance,
and there's no learning signal.

In [ ]:
def completion_text(c):
    return c[0]["content"] if isinstance(c, list) else c


def correctness_reward(completions, target, **kwargs):
    rewards = []
    for comp, t in zip(completions, target):
        s = score(completion_text(comp), json.loads(t))
        rewards.append(0.2 * s["valid_json"] + 0.25 * s["date"] + 0.25 * s["amount"] + 0.3 * s["currency"])
    return rewards


def format_reward(completions, **kwargs):
    """Bonus for answering with JSON *only* (no chatty wrapper)."""
    return [0.2 if completion_text(c).strip().startswith("{") and completion_text(c).strip().endswith("}") else 0.0 for c in completions]


t = dataset(1, seed=0)[0]["target"]
print(correctness_reward([[{"content": json.dumps(t)}], [{"content": "no idea"}]], [json.dumps(t)] * 2), format_reward([[{"content": json.dumps(t)}]]))

## 2. Train
Prompts only (no answers!) + the target the *checker* needs. Small settings so it runs on a laptop in minutes: 4 samples per prompt,
short completions. Generation dominates the time — production RL uses vLLM for sampling.

In [ ]:
train = Dataset.from_list([{"prompt": messages(e["text"]), "target": json.dumps(e["target"])} for e in dataset(400, seed=21)])
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32).to(device)


def evaluate(m):
    m.eval()
    s = []
    for e in test:
        ids = tok.apply_chat_template(messages(e["text"]), add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(m.device)
        with torch.no_grad():
            out = m.generate(ids, max_new_tokens=48, do_sample=False, pad_token_id=tok.eos_token_id)
        s.append(score(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True), e["target"]))
    return {k: round(sum(x[k] for x in s) / len(s), 2) for k in s[0]}


before = evaluate(model)
print("before:", before)

config = GRPOConfig(
    output_dir=str(FT / "grpo-run"), max_steps=60, learning_rate=5e-4,
    per_device_train_batch_size=8, num_generations=4,  # 2 prompts x 4 samples per step
    max_completion_length=48, temperature=1.0, beta=0.0,  # beta=0: no KL penalty (common in recent recipes)
    logging_steps=5, save_strategy="no", report_to="none",
)
trainer = GRPOTrainer(model=model, args=config, train_dataset=train, processing_class=tok, reward_funcs=[correctness_reward, format_reward],
                      peft_config=LoraConfig(r=16, lora_alpha=32, target_modules="all-linear", task_type="CAUSAL_LM"))
start = time.perf_counter()
trainer.train()
print(f"GRPO: {time.perf_counter() - start:.0f}s")

## 3. Reward curves

In [ ]:
hist = [h for h in trainer.state.log_history if "reward" in h]
steps = [h["step"] for h in hist]
plt.plot(steps, [h["reward"] for h in hist], label="mean total reward")
for key in [k for k in hist[0] if k.startswith("rewards/") and k.endswith("/mean")]:
    plt.plot(steps, [h[key] for h in hist], "--", label=key.split("/")[1])
plt.xlabel("step"); plt.legend(); plt.title("GRPO on payment extraction"); plt.show()
print("after:", evaluate(trainer.model))

Compare with 12.03: SFT reached near-perfect exact match in a similar time. **For tasks where you can write the answer, SFT is cheaper and
stronger.** RLVR earns its keep where answers are checkable but not easily written down — math with long derivations, code that must pass tests,
multi-step agents — and as a stage *after* SFT to sharpen and generalise.

## 4. Failure modes
- **Reward hacking**: the model finds what the checker rewards, not what you meant. E.g. if `format_reward` only checked `startswith("{")`, outputs
  like `{}` would collect format points forever. Inspect samples, not just the curve.
- **Zero variance**: all samples in a group get the same reward → advantage 0 → no gradient. Use partial credit, raise temperature, filter prompts
  that are too easy/hard (DAPO-style dynamic sampling).
- **Length blow-ups** in reasoning RL: cap `max_completion_length`, consider length penalties.
- **Speed**: generation is the bottleneck; use vLLM (`use_vllm=True` on CUDA) and bigger batches.

## Exercises
1. Start GRPO from the 12.03 SFT adapter instead of the base model. Does RL improve the held-out templates further?
2. Make a reasoning task: "What is the total of these three payments?" with the sum as the checkable answer and a `<think>...</think><answer>...</answer>`
   format reward. Run it on Colab with Unsloth's GRPO notebook and Qwen3-1.7B.
3. Deliberately write a hackable reward and watch the model exploit it. Then fix the reward.

In [ ]:
# your code here